In [ ]:
import pandas as pd
df=pd.read_excel("/content/rideBookings_preprocessed1.xlsx")
df.head(3)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Customer Rating,Payment Method,Datetime,Year,Month,Day,DayOfWeek,Hour,WeekOfYear,IsWeekend
0,2024-01-01,18:05:08,CNR2589191,Completed,CID8552101,Auto,Jahangirpuri,Saket,3.8,20.1,...,4.5,Uber Wallet,2024-01-01 18:05:08,2024,1,1,0,18,1,0
1,2024-01-01,18:39:09,CNR6304210,Completed,CID7004065,Uber XL,IGI Airport,Udyog Bhawan,7.5,33.2,...,4.6,UPI,2024-01-01 18:39:09,2024,1,1,0,18,1,0
2,2024-01-01,17:26:26,CNR4423001,Cancelled by Driver,CID6188095,Bike,IGNOU Road,Bhikaji Cama Place,7.8,NaN,...,NaN,UPI,2024-01-01 17:26:26,2024,1,1,0,17,1,0


In [ ]:
!pip install pandasql


  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=f2dd316ee0f9bd03fda3a3457b48899d14e67d5753641f0f551d024457907172
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [ ]:
from pandasql import sqldf
import duckdb
!pip install duckdb openpyxl --quiet
df.columns = df.columns.str.replace(' ', '')

Top 10 pickup locations by total rides

In [ ]:
q1 = duckdb.query("""
    SELECT
        PickupLocation,
        COUNT(*) AS total_rides,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation
    ORDER BY total_rides DESC
    LIMIT 10
""").df()
print("\n── Q1: Top 10 Pickup Locations ──")
print(q1.to_string(index=False))


── Q1: Top 10 Pickup Locations ──
  PickupLocation  total_rides  pct_of_total
         Khandsa          599          0.65
 Barakhamba Road          591          0.64
   Subhash Chowk          575          0.62
         Madipur          571          0.62
        Mehrauli          567          0.61
  Kanhaiya Nagar          566          0.61
 Ashok Park Main          565          0.61
Dwarka Sector 21          563          0.61
        Badarpur          562          0.61
 Lok Kalyan Marg          560          0.61


Top 10 drop locations by total rides

In [ ]:
q2 = duckdb.query("""
    SELECT
        DropLocation,
        COUNT(*) AS total_rides,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY DropLocation
    ORDER BY total_rides DESC
    LIMIT 10
""").df()
print("\n── Q2: Top 10 Drop Locations ──")
print(q2.to_string(index=False))


── Q2: Top 10 Drop Locations ──
      DropLocation  total_rides  pct_of_total
            Ashram          588          0.64
       Preet Vihar          584          0.63
         Sultanpur          580          0.63
   Noida Extension          574          0.62
      Lajpat Nagar          571          0.62
       Narsinghpur          571          0.62
   Sarai Kale Khan          570          0.62
         Cyber Hub          568          0.62
        Dwarka Mor          567          0.61
Kashmere Gate ISBT          567          0.61


Vehicle type preference by pickup location

In [ ]:
q3 = duckdb.query("""
    SELECT
        PickupLocation,
        VehicleType,
        COUNT(*) AS bookings,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY PickupLocation), 2) AS share_pct
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation, VehicleType
    ORDER BY PickupLocation, bookings DESC
""").df()
print("\n── Q3: Vehicle Type Preference by Pickup Location ──")
print(q3.to_string(index=False))


── Q3: Vehicle Type Preference by Pickup Location ──
           PickupLocation   VehicleType  bookings  share_pct
                    AIIMS          Auto       161      28.90
                    AIIMS      Go Sedan       101      18.13
                    AIIMS       Go Mini       100      17.95
                    AIIMS          Bike        79      14.18
                    AIIMS Premier Sedan        64      11.49
                    AIIMS         eBike        33       5.92
                    AIIMS       Uber XL        19       3.41
             Adarsh Nagar          Auto       142      26.84
             Adarsh Nagar       Go Mini       113      21.36
             Adarsh Nagar      Go Sedan        88      16.64
             Adarsh Nagar          Bike        74      13.99
             Adarsh Nagar Premier Sedan        64      12.10
             Adarsh Nagar         eBike        32       6.05
             Adarsh Nagar       Uber XL        16       3.02
               Akshardham      

Cancellation rate by pickup location

In [ ]:
q4 = duckdb.query("""
    SELECT
        PickupLocation,
        COUNT(*) AS total_bookings,
        SUM(CASE WHEN BookingStatus = 'Cancelled by Driver'   THEN 1 ELSE 0 END) AS driver_cancels,
        SUM(CASE WHEN BookingStatus = 'Cancelled by Customer' THEN 1 ELSE 0 END) AS customer_cancels,
        ROUND(
            (SUM(CASE WHEN BookingStatus IN (
                'Cancelled by Driver', 'Cancelled by Customer') THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*)), 2
        ) AS cancel_rate_pct
    FROM df
    GROUP BY PickupLocation
    HAVING total_bookings >= 20
    ORDER BY cancel_rate_pct DESC
    LIMIT 15
""").df()
print("\n── Q4: Cancellation Rate by Pickup Location ──")
print(q4.to_string(index=False))


── Q4: Cancellation Rate by Pickup Location ──
      PickupLocation  total_bookings  driver_cancels  customer_cancels  cancel_rate_pct
          Vinobapuri             818           176.0              66.0            29.58
          Akshardham             835           169.0              78.0            29.58
         Qutub Minar             816           172.0              61.0            28.55
             Munirka             811           171.0              59.0            28.36
            Kadarpur             834           169.0              65.0            28.06
 Faridabad Sector 15             825           163.0              67.0            27.88
         Nehru Place             880           190.0              53.0            27.61
            Shahdara             815           167.0              58.0            27.61
          Chhatarpur             826           173.0              55.0            27.60
          Karkarduma             828           168.0              60.0  

Top driver cancellation reasons by area

In [ ]:
q5 = duckdb.query("""
    SELECT
        PickupLocation,
        DriverCancellationReason,
        COUNT(*) AS cancellations
    FROM df
    WHERE BookingStatus = 'Cancelled by Driver'
      AND DriverCancellationReason IS NOT NULL
    GROUP BY PickupLocation, DriverCancellationReason
    ORDER BY PickupLocation, cancellations DESC
""").df()
print("\n── Q5: Driver Cancellation Reasons by Area ──")
print(q5.to_string(index=False))


── Q5: Driver Cancellation Reasons by Area ──
           PickupLocation            DriverCancellationReason  cancellations
                    AIIMS      The customer was coughing/sick             49
                    AIIMS More than permitted people in there             47
                    AIIMS       Personal & Car related issues             42
                    AIIMS              Customer related issue             35
             Adarsh Nagar      The customer was coughing/sick             40
             Adarsh Nagar More than permitted people in there             39
             Adarsh Nagar       Personal & Car related issues             37
             Adarsh Nagar              Customer related issue             24
               Akshardham      The customer was coughing/sick             52
               Akshardham More than permitted people in there             48
               Akshardham              Customer related issue             43
               Akshardham    

Locations with highest 'No Driver Found' rate

In [ ]:
q6 = duckdb.query("""
    SELECT
        PickupLocation,
        COUNT(*) AS total_bookings,
        SUM(CASE WHEN BookingStatus = 'No Driver Found' THEN 1 ELSE 0 END) AS no_driver_count,
        ROUND(
            SUM(CASE WHEN BookingStatus = 'No Driver Found' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        ) AS no_driver_rate_pct
    FROM df
    GROUP BY PickupLocation
    HAVING total_bookings >= 15
    ORDER BY no_driver_rate_pct DESC
    LIMIT 15
""").df()
print("\n── Q6: No Driver Found Rate by Pickup Location ──")
print(q6.to_string(index=False))


── Q6: No Driver Found Rate by Pickup Location ──
      PickupLocation  total_bookings  no_driver_count  no_driver_rate_pct
         Old Gurgaon             803             83.0               10.34
           Paharganj             816             78.0                9.56
          Vinobapuri             818             75.0                9.17
       Greater Noida             859             77.0                8.96
       Pataudi Chowk             901             80.0                8.88
         Rohini West             828             73.0                8.82
           Arjangarh             805             70.0                8.70
          Dwarka Mor             837             72.0                8.60
           Pitampura             846             72.0                8.51
        Udyog Bhawan             835             71.0                8.50
Netaji Subhash Place             823             69.0                8.38
  Bhikaji Cama Place             817             68.0        

Revenue by pickup location

In [ ]:
q7 = duckdb.query("""
    SELECT
        PickupLocation,
        COUNT(*) AS completed_rides,
        SUM(BookingValue) AS total_revenue,
        ROUND(AVG(BookingValue), 2) AS avg_fare,
        ROUND(AVG(RideDistance), 2) AS avg_distance_km
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation
    ORDER BY total_revenue DESC
    LIMIT 15
""").df()
print("\n── Q7: Revenue by Pickup Location ──")
print(q7.to_string(index=False))


── Q7: Revenue by Pickup Location ──
  PickupLocation  completed_rides  total_revenue  avg_fare  avg_distance_km
 Barakhamba Road              591       299895.0    507.44            24.87
         Khandsa              599       297632.5    496.88            26.09
   Pataudi Chowk              551       289847.5    526.04            25.78
     Tughlakabad              544       288284.0    529.93            25.29
   Subhash Chowk              575       286965.5    499.07            25.79
        Inderlok              557       284131.0    510.11            26.02
           AIIMS              557       282422.5    507.04            26.07
 Ashok Park Main              565       281963.5    499.05            25.98
      Peeragarhi              549       280834.5    511.54            25.78
        Badarpur              562       280613.0    499.31            25.80
Dwarka Sector 21              563       279439.5    496.34            26.43
  Samaypur Badli              554       279406.5  

Revenue per km by location corridor

In [ ]:
q8 = duckdb.query("""
    SELECT
        PickupLocation,
        DropLocation,
        COUNT(*) AS trip_count,
        ROUND(SUM(BookingValue) / NULLIF(SUM(RideDistance), 0), 2) AS revenue_per_km
    FROM df
    WHERE BookingStatus = 'Completed'
      AND RideDistance > 0
    GROUP BY PickupLocation, DropLocation
    HAVING trip_count >= 10
    ORDER BY revenue_per_km DESC
    LIMIT 15
""").df()
print("\n── Q8: Revenue per KM by Corridor ──")
print(q8.to_string(index=False))


── Q8: Revenue per KM by Corridor ──
 PickupLocation        DropLocation  trip_count  revenue_per_km
    Indirapuram          Akshardham          10           32.07
        Rithala Udyog Vihar Phase 4          11           30.49
       Ghitorni         Mandi House          10           26.90
   Indraprastha      Rajouri Garden          10           26.70
  Ambience Mall          Akshardham          11           23.75
      Cyber Hub       Ambience Mall          10           23.58
    Sushant Lok     Sarai Kale Khan          10           23.45
    Bahadurgarh         Jama Masjid          10           22.75
    Jama Masjid         Khan Market          10           22.66
      Paharganj      Sarojini Nagar          10           22.63
Noida Extension        Vidhan Sabha          10           21.83
Lok Kalyan Marg             Jhilmil          10           21.77
      Moolchand         Anand Vihar          10           20.23
          AIIMS               Saket          10           19.77
  

Incomplete ride rate by pickup location

In [ ]:
q9 = duckdb.query("""
    SELECT
        PickupLocation,
        COUNT(*) AS total_rides,
        SUM(CASE WHEN BookingStatus = 'Incomplete' THEN 1 ELSE 0 END) AS incomplete_count,
        ROUND(
            SUM(CASE WHEN BookingStatus = 'Incomplete' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        ) AS incomplete_rate_pct
    FROM df
    GROUP BY PickupLocation
    HAVING total_rides >= 20
    ORDER BY incomplete_rate_pct DESC
    LIMIT 15
""").df()
print("\n── Q9: Incomplete Ride Rate by Pickup Location ──")
print(q9.to_string(index=False))


── Q9: Incomplete Ride Rate by Pickup Location ──
 PickupLocation  total_rides  incomplete_count  incomplete_rate_pct
 Kanhaiya Nagar          885              72.0                 8.14
      Lal Quila          824              67.0                 8.13
      Cyber Hub          856              66.0                 7.71
          Okhla          818              62.0                 7.58
    Udyog Vihar          888              67.0                 7.55
      Kaushambi          848              63.0                 7.43
    Narsinghpur          836              62.0                 7.42
   Indraprastha          845              62.0                 7.34
     Akshardham          835              61.0                 7.31
South Extension          834              60.0                 7.19
    Mayur Vihar          860              61.0                 7.09
    Tilak Nagar          889              63.0                 7.09
  Saket A Block          819              58.0                 7.

Most frequent pickup-drop location pairs

In [ ]:
q10 = duckdb.query("""
    SELECT
        PickupLocation,
        DropLocation,
        COUNT(*) AS trip_count,
        ROUND(AVG(RideDistance), 2) AS avg_distance_km,
        ROUND(AVG(BookingValue), 2) AS avg_fare
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation, DropLocation
    ORDER BY trip_count DESC
    LIMIT 15
""").df()
print("\n── Q10: Top Pickup-Drop Corridors ──")
print(q10.to_string(index=False))



── Q10: Top Pickup-Drop Corridors ──
  PickupLocation        DropLocation  trip_count  avg_distance_km  avg_fare
  DLF City Court             Bhiwadi          13            30.76    455.77
     Rohini West          Sohna Road          13            29.41    477.23
   Ambience Mall          Akshardham          11            26.94    639.77
 Noida Sector 62     Sarai Kale Khan          11            24.93    465.82
         Rithala Udyog Vihar Phase 4          11            18.86    575.00
   Subhash Chowk          IGNOU Road          11            28.02    469.09
 Vishwavidyalaya         Yamuna Bank          10            35.86    304.70
           Saket          Karkarduma          10            24.48    319.10
  Kanhaiya Nagar            Vaishali          10            24.73    456.20
Dwarka Sector 21          Govindpuri          10            31.51    574.80
 Noida Extension        Vidhan Sabha          10            20.70    451.85
     Anand Vihar            RK Puram          10  

Peak demand hours by pickup location

In [ ]:
q11 = duckdb.query("""
    SELECT
        PickupLocation,
        Hour,
        COUNT(*) AS ride_count
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation, Hour
    ORDER BY PickupLocation, ride_count DESC
""").df()
print("\n── Q11: Peak Demand Hours by Pickup Location ──")
print(q11.to_string(index=False))


── Q11: Peak Demand Hours by Pickup Location ──
           PickupLocation  Hour  ride_count
                    AIIMS    17          49
                    AIIMS    19          46
                    AIIMS    18          45
                    AIIMS    20          37
                    AIIMS    16          34
                    AIIMS    21          32
                    AIIMS    11          32
                    AIIMS     9          30
                    AIIMS    10          30
                    AIIMS    14          30
                    AIIMS    15          27
                    AIIMS    12          24
                    AIIMS     8          24
                    AIIMS    13          20
                    AIIMS    22          18
                    AIIMS     6          15
                    AIIMS     7          14
                    AIIMS     5          12
                    AIIMS    23          11
                    AIIMS     4           9
                    AIIMS  

Weekend vs weekday demand by location

In [ ]:
q12= duckdb.query("""
    SELECT
        PickupLocation,
        SUM(CASE WHEN IsWeekend = 1 THEN 1 ELSE 0 END) AS weekend_rides,
        SUM(CASE WHEN IsWeekend = 0 THEN 1 ELSE 0 END) AS weekday_rides,
        ROUND(
            SUM(CASE WHEN IsWeekend = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        ) AS weekend_pct
    FROM df
    WHERE BookingStatus = 'Completed'
    GROUP BY PickupLocation
    HAVING (weekend_rides + weekday_rides) >= 20
    ORDER BY weekend_pct DESC
    LIMIT 15
""").df()
print("\n── Q12: Weekend vs Weekday Demand by Location ──")
print(q12.to_string(index=False))


── Q12: Weekend vs Weekday Demand by Location ──
         PickupLocation  weekend_rides  weekday_rides  weekend_pct
    Civil Lines Gurgaon          177.0          351.0        33.52
        Sarai Kale Khan          173.0          351.0        33.02
                Bhiwadi          170.0          347.0        32.88
              Sultanpur          174.0          360.0        32.58
            Tughlakabad          177.0          367.0        32.54
Gurgaon Railway Station          176.0          370.0        32.23
        Noida Sector 18          161.0          339.0        32.20
               Jor Bagh          173.0          366.0        32.10
       Hero Honda Chowk          167.0          355.0        31.99
             Moti Nagar          173.0          370.0        31.86
             India Gate          175.0          376.0        31.76
                  Okhla          166.0          357.0        31.74
     Bhikaji Cama Place          155.0          335.0        31.63
            